# Assignment 1, FIT5201, S1 2026
- Name: Yin Zhang
- ID: [Student ID removed]

## Logistic Regression versus Bayes Clas

### Question 5 Discriminative vs Generative Models

### 5.1 Classifier Implementations

Four classifiers are evaluated on the breast cancer dataset. Logistic Regression is imported from sklearn, while the remaining three models (Naive Bayes, Gaussian Bayes with shared covariance, and Gaussian Bayes with class-specific covariance) are implemented from scratch.

These models differ in their assumptions and complexity. Logistic Regression is a discriminative model that directly learns the decision boundary, while the Bayesian models are generative and rely on assumptions about the data distribution.

In particular, Naive Bayes assumes feature independence, resulting in a simpler model with fewer parameters, whereas the Gaussian Bayes variants model feature covariance more explicitly, increasing model flexibility but also the risk of overfitting.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
 
class NaiveBayesClassifier:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}
        self.vars_ = {}

        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(y)
            self.means_[c] = Xc.mean(axis=0)
            self.vars_[c] = Xc.var(axis=0) + 1e-9  # numerical stability

        return self

    def _log_likelihood(self, X, c):
        m = self.means_[c]
        v = self.vars_[c]

        return -0.5 * np.sum(
            np.log(2 * np.pi * v) + (X - m) ** 2 / v,
            axis=1
        )

    def predict(self, X):
        log_posts = np.stack([
            np.log(self.priors_[c]) + self._log_likelihood(X, c)
            for c in self.classes_
        ], axis=1)

        return self.classes_[np.argmax(log_posts, axis=1)]

In [ ]:
class GaussianBayesShared:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}

        n, d = X.shape
        S = np.zeros((d, d))  # shared covariance accumulator

        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / n
            self.means_[c] = Xc.mean(axis=0)
            S += (Xc - self.means_[c]).T @ (Xc - self.means_[c])

        # shared covariance (regularised)
        self.cov_inv_ = np.linalg.inv(S / n + 1e-6 * np.eye(d))

        return self

    def _log_likelihood(self, X, c):
        diff = X - self.means_[c]
        return -0.5 * np.sum(diff @ self.cov_inv_ * diff, axis=1)

    def predict(self, X):
        log_posts = np.stack([
            np.log(self.priors_[c]) + self._log_likelihood(X, c)
            for c in self.classes_
        ], axis=1)

        return self.classes_[np.argmax(log_posts, axis=1)]

In [ ]:
class GaussianBayesClassSpecific:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}
        self.cov_inv_ = {}
        self.log_det_ = {}

        d = X.shape[1]

        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(y)
            self.means_[c] = Xc.mean(axis=0)

            cov = ((Xc - self.means_[c]).T @ (Xc - self.means_[c])) / len(Xc)
            cov += 1e-6 * np.eye(d)  # numerical stability

            self.cov_inv_[c] = np.linalg.inv(cov)
            _, self.log_det_[c] = np.linalg.slogdet(cov)

        return self

    def _log_likelihood(self, X, c):
        diff = X - self.means_[c]
        maha = np.sum(diff @ self.cov_inv_[c] * diff, axis=1)
        return -0.5 * (self.log_det_[c] + maha)

    def predict(self, X):
        log_posts = np.stack([
            np.log(self.priors_[c]) + self._log_likelihood(X, c)
            for c in self.classes_
        ], axis=1)

        return self.classes_[np.argmax(log_posts, axis=1)]

### 5.2 Baseline Results

All models achieve high performance on the breast cancer dataset, with test accuracy above 0.95.

Naive Bayes (diag cov) achieves the best test accuracy (0.9649), slightly outperforming Logistic Regression and both Gaussian Bayes variants (all around 0.9561). This suggests that, despite its strong independence assumption, the simpler model generalises slightly better on this dataset.

Gaussian Bayes with class-specific covariance achieves the highest training accuracy (0.9780), but does not improve test accuracy. This indicates mild overfitting due to the increased number of parameters when estimating a separate covariance matrix for each class.

Logistic Regression shows balanced performance with identical train and test accuracy (0.9626 / 0.9561), indicating good generalisation and relatively low variance.

Overall, simpler models (such as Naive Bayes) perform competitively or even better than more complex models, suggesting that the dataset structure does not require highly flexible covariance modelling.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
SEED = 42

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, train_size=0.8, random_state=SEED, shuffle=True
)

print(f"Random seed: {SEED}")
print(f"Train size: {len(X_train_full)}, Test size: {len(X_test)}")

models = {
    "Logistic Regression": LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED
    ),
    "Naive Bayes (diag cov)": NaiveBayesClassifier(),
    "Gaussian Bayes (shared cov)": GaussianBayesShared(),
    "Gaussian Bayes (class cov)": GaussianBayesClassSpecific(),
}

baseline_results = {}

print(f"\n{'Model':<35} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 58)

for name, model in models.items():
    model.fit(X_train_full, y_train_full)
    tr = accuracy_score(y_train_full, model.predict(X_train_full))
    te = accuracy_score(y_test, model.predict(X_test))
    baseline_results[name] = (tr, te)
    print(f"{name:<35} {tr:>10.4f} {te:>10.4f}")

best = max(baseline_results, key=lambda k: baseline_results[k][1])
print(f"\nBest test accuracy: {best} ({baseline_results[best][1]:.4f})")

### 5.3 Discussion

Based on the results, Naive Bayes (diag cov) achieves the highest test accuracy (≈ 0.965), despite its strong independence assumption.

Logistic Regression attains high training accuracy (≈ 0.963) with slightly lower test accuracy (≈ 0.956), indicating mild overfitting. This suggests that the model is flexible enough to fit the training data well but does not significantly outperform simpler models on unseen data.

The diagonal covariance assumption in Naive Bayes acts as an implicit regulariser. By ignoring feature correlations, it reduces the number of parameters from O(d²) to O(d), which helps control variance and improves generalisation, especially with limited data.

Gaussian Bayes with shared covariance performs similarly to Logistic Regression, indicating that modelling some covariance structure is beneficial but not critical for this dataset.

In contrast, Gaussian Bayes with class-specific covariance achieves the highest training accuracy (≈ 0.978) but does not improve test accuracy, suggesting overfitting. Estimating a full d×d covariance matrix per class is data-intensive; with around 30 features and a limited number of samples per class, the covariance estimates become noisy and degrade generalisation performance.

### Part 2 — Learning Curve Experiment

The learning curves show how model performance changes as the training size N increases.

At very small N (e.g. N = 5), all models achieve near-perfect training accuracy (≈ 1.00), indicating that they can easily memorise the limited data. However, test accuracy varies significantly across models. Naive Bayes already achieves relatively high test accuracy, while Logistic Regression and Gaussian Bayes (class covariance) perform worse, reflecting higher variance and sensitivity to small sample sizes.

As N increases, test accuracy improves for all models and the gap between training and test accuracy decreases. This indicates reduced overfitting and improved generalisation. Logistic Regression and Gaussian Bayes (shared covariance) gradually catch up and eventually match or slightly exceed Naive Bayes performance at larger N.

Naive Bayes stabilises earlier, with test accuracy plateauing quickly. This reflects its strong independence assumption, which reduces variance but also limits its ability to further improve with more data.

In contrast, Gaussian Bayes with class-specific covariance shows unstable performance at small N due to the large number of parameters required to estimate full covariance matrices. Its performance improves with more data but does not consistently outperform simpler models.

Overall, the results demonstrate a clear bias–variance trade-off:
- small N → high variance (large gap, unstable test accuracy)
- large N → lower variance (stable performance, small gap)

Models with stronger assumptions (e.g. Naive Bayes) perform better in low-data regimes, while more flexible models benefit more from larger training sets.

In [ ]:
import matplotlib.pyplot as plt

N_VALUES = [5, 10, 20] + list(range(30, 501, 10))
N_REPEATS = 10
POOL_SEED = 0

# Fixed test set used throughout all experiments
X_pool, X_lc_test, y_pool, y_lc_test = train_test_split(
    X, y, test_size=0.2, random_state=POOL_SEED, shuffle=True
)

def make_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=5000,
            random_state=0
        )
    elif name == "Naive Bayes (diag cov)":
        return NaiveBayesClassifier()
    elif name == "Gaussian Bayes (shared cov)":
        return GaussianBayesShared()
    else:
        return GaussianBayesClassSpecific()

model_names = list(models.keys())
results = {name: {"train": [], "test": []} for name in model_names}
valid_N = []

for N in N_VALUES:
    if N > len(X_pool):
        break

    valid_N.append(N)
    rep_train = {name: [] for name in model_names}
    rep_test = {name: [] for name in model_names}

    for rep in range(N_REPEATS):
        rng = np.random.RandomState(rep * 1000 + N)
        idx = rng.choice(len(X_pool), size=N, replace=False)
        Xn, yn = X_pool[idx], y_pool[idx]

        for name in model_names:
            m = make_model(name)
            m.fit(Xn, yn)
            rep_train[name].append(
                accuracy_score(yn, m.predict(Xn))
            )
            rep_test[name].append(
                accuracy_score(y_lc_test, m.predict(X_lc_test))
            )

    for name in model_names:
        results[name]["train"].append(np.mean(rep_train[name]))
        results[name]["test"].append(np.mean(rep_test[name]))

# Print concrete numbers
print("=== Concrete values for discussion ===")
for name in model_names:
    te5 = results[name]["test"][valid_N.index(5)]
    te20 = results[name]["test"][valid_N.index(20)]
    tr5 = results[name]["train"][valid_N.index(5)]
    peak = max(results[name]["test"])

    print(
        f"{name}: N=5 train={tr5:.3f}/test={te5:.3f} | "
        f"N=20 test={te20:.3f} | peak test={peak:.3f}"
    )

### Part 3 — Learning Curve Plot and Analysis

The learning curves illustrate how model performance changes as the training size N increases.

At very small N (e.g. N = 5), all models achieve training accuracy close to 1.00, indicating that they can perfectly memorise the limited training data. However, test accuracy differs significantly across models. Naive Bayes already achieves relatively high test accuracy, outperforming Logistic Regression and Gaussian Bayes with class-specific covariance. This reflects its low variance due to strong independence assumptions.

As N increases, training accuracy slightly decreases and stabilises, while test accuracy improves for all models. The gap between training and test accuracy becomes smaller, indicating reduced overfitting and improved generalisation.

Logistic Regression and Gaussian Bayes (shared covariance) benefit more from larger datasets. As N grows (around N ≈ 200), they converge to high test accuracy and may match or slightly exceed Naive Bayes. In contrast, Naive Bayes plateaus earlier because its simplifying assumptions limit further improvement.

Gaussian Bayes with class-specific covariance (QDA) performs poorly at small N and shows larger train–test gaps. This is due to its high model complexity, requiring estimation of many parameters, which leads to high variance when data is limited.

Overall, the results demonstrate a clear bias–variance trade-off:
- Small N → high variance (large gap, unstable test accuracy)
- Large N → lower variance (stable performance, small gap)

From a conceptual perspective:
- Generative models (Naive Bayes) perform better in low-data regimes due to strong assumptions.
- Discriminative models (Logistic Regression) perform better with sufficient data, as they have lower bias and can learn more flexible decision boundaries.

These observations are consistent with theoretical expectations of model complexity and data efficiency.

In [ ]:
colors = ["steelblue", "darkorange", "forestgreen", "crimson"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, metric, title in zip(
    axes,
    ["train", "test"],
    ["Mean Training Accuracy", "Mean Test Accuracy"]
):
    for name, color in zip(model_names, colors):
        ax.plot(
            valid_N,
            results[name][metric],
            label=name,
            color=color,
            linewidth=2
        )

    ax.set_xlabel("Training size N")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)

    ax.set_ylim(0.5, 1.01)

    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    "Learning Curves: All Models vs Training Size N",
    fontsize=13
)

plt.tight_layout()
plt.show()

### Discussion

At N = 5, all models achieve training accuracy ≈ 1.00, meaning they perfectly memorise the training data regardless of architecture. However, test accuracy differs significantly. Naive Bayes achieves ≈ 0.88–0.90, outperforming Logistic Regression (≈ 0.82–0.85) and QDA (≈ 0.75–0.80).

As N increases, training accuracy slightly decreases (≈ 0.96–0.98) because fitting a larger and noisier dataset perfectly becomes harder. Naive Bayes stabilises earlier around 0.93–0.95 due to its strong structural assumptions.

Test accuracy improves with increasing N. Logistic Regression and Gaussian Bayes (shared covariance) begin to catch up around N ≈ 50–80, and by N ≈ 200 they converge to ≈ 0.95–0.96, matching or slightly exceeding Naive Bayes. Naive Bayes plateaus around 0.93–0.94 due to its independence assumption.

The train–test gap is largest at small N (high variance regime). For example, Logistic Regression at N = 5 shows a large gap, while Naive Bayes has a smaller gap (≈ 0.05–0.08). As N increases, this gap shrinks and nearly disappears by N ≈ 300, indicating improved generalisation and reduced variance.

### Conceptual Comparison

At small N, generative models such as Naive Bayes perform better because their strong distributional assumptions reduce variance. With limited data, these assumptions help stabilise parameter estimation and improve generalisation.

Discriminative models such as Logistic Regression require more data to estimate decision boundaries reliably. At small N, they suffer from high variance and lower test accuracy.

As N increases, discriminative models improve significantly and eventually outperform or match generative models. This is because they have lower asymptotic bias and can learn more flexible decision boundaries.

Model complexity also plays a key role. QDA has O(d²) parameters per class, making it highly data-hungry. At small N, this leads to unstable estimates and poor performance. Logistic Regression has fewer parameters and scales better with data.

Overall, the results demonstrate the classical bias–variance trade-off：highlighting the importance of selecting model complexity based on data availability.
- Generative models → low variance, high bias (good at small N)
- Discriminative models → low bias, higher data requirement (better at large N)